### Librerias e importaciones

In [47]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from src.preprocessing import preprocesamiento_pre_split, preprocesamiento_post_split, onehot_encoding
from src.data_splitting import train_val_split
from src.plots import eda_visualizacion_suvs, plot_precio_segun_antiguedad_km, plot_precio_segun_rango_ant, plot_dispersion_por_marca, plot_grupo_especifico
from src.modelos.xgboost import entrenar_xgboost, entrenar_xgboost_ohe, grid_search
from src.modelos.regresion_lineal import entrenar_regresion_lineal, definir_regularizacion
from src.utils import estandarizar
from src.deteccion_outliers import flagear_outliers_por_grupo, reportar_outliers_por_grupo, ver_outliers, eliminar_outliers_grupo, eliminar_outliers_por_corte

: 

In [ ]:
#IGNORAR ESTA CELDA
%load_ext autoreload
%autoreload 2

: 

### EDA

**Preprocessing**

In [ ]:
data = pd.read_csv('../data/raw/pf_suvs.csv')

: 

In [ ]:
print("\n Dataset ALEATORIO")
print(data.sample(5))

print("\n -------- RESUMEN - ESTADISTICOS E INFO -------- \n" )
print("\n DATA INFO")
print(data.info())

: 

In [ ]:
summary = pd.DataFrame({
    "dtype": data.dtypes,
    "nulos": data.isnull().sum(),
    "unicos": data.nunique()
})
summary

: 

In [ ]:
print("\n DATA DESCRIPCION " )
data.describe()

: 

<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
<em>El análisis exploratorio permitió detectar la existencia de valores faltantes en diversas variables descriptivas del vehículo, entre ellas Color, Transmisión, Motor y Con cámara de retroceso. Debido a que estas variables pueden influir significativamente en el valor de mercado de una SUV, se optó por imputar los datos faltantes en lugar de descartar las observaciones correspondientes, preservando así el tamaño de la muestra disponible para entrenamiento.</em>
</p>

<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
<em>Por otra parte, se observó que gran parte de los atributos del dataset se encuentran representados mediante variables categóricas de tipo texto. Dado que los algoritmos de aprendizaje supervisado utilizados trabajan sobre representaciones numéricas, será necesario transformar dichas variables mediante One-Hot Encoding, generando una codificación binaria que permita incorporar esta información al modelo sin introducir relaciones artificiales entre categorías.<em>
</p>

***PREPROCESAMIENTO: Limpieza de datos necesaria para no afectar el entrenamiento del modelo.***

**Justificacion de desiciones tomadas**


<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
1. <code>Unnamed: 0</code> actúa únicamente como identificador de cada publicación y no aporta información relevante para la predicción del precio, por lo que fue eliminada.
Para <code>Tipo de carrocería</code>, el dataset contiene exclusivamente vehículos SUV, por lo que también fue eliminada.</p>

<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
2. <code>Cambio de moneda a USD</code>: Al tratarse de la variable objetivo del problema, fue necesario unificar todas las observaciones en una única moneda (USD) de referencia para garantizar la comparabilidad entre vehículos. Se eliminó la columna ya que no aportaba información.</p>

<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
3. <code>Título</code>: Gran parte de la información contenida en esta columna ya se encuentra representada en otras variables estructuradas del dataset (<code>Marca</code>, <code>Modelo</code>, <code>Motor</code>, <code>Versión</code>, etc.), por lo que se decidió eliminarla para evitar redundancia.</p>

<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
4. <code>Imputación de Puertas</code>: se observaron valores inconsistentes en la variable <code>Puertas</code>. Dado que el conjunto de datos contiene exclusivamente SUVs, se decidió homogeneizar los valores restantes utilizando cinco puertas como valor por defecto.</p>


In [ ]:
display(data["Puertas"].value_counts().sort_index())

: 

<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
<p>5. <code>Motor</code>: presentaba únicamente 38 valores faltantes (menos del 0.3% del dataset). Debido a su baja proporción, se optó por eliminar dichas observaciones en lugar de imputarlas para evitar introducir ruido artificial. A su vez, los datos estaban en <code>str</code> se cambiaron a valor numérico.</p>
<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
<p>Además la columna contenía formatos muy heterogéneos, combinando cilindrada, tipo de motor y descripciones comerciales (por ejemplo: <em>"2.0 TSI"</em>, <em>"1.5 Turbo"</em>, <em>"Inyección Multipunto"</em> o <em>"Turboalimentado"</em>).</p>
<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
<p>Para preservar la mayor cantidad de información posible se descompuso en varias variables:
<strong>Motor_Litros</strong>, <strong>Motor_Turbo</strong>, <strong>Motor_Multipunto</strong>, <strong>Motor_Diesel</strong>, <strong>Motor_Hibrido</strong>, <strong>Motor_Litros_Faltante</strong>.</p>
<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
<p>De esta forma se evita descartar observaciones y se conserva información relevante sobre la motorización del vehículo en un formato apto para modelos supervisados.</p>


In [ ]:
nulos_motor = data["Motor"].isna().sum()
total = len(data)

print(f"Nulos en Motor: {nulos_motor}")
print(f"Total de registros: {total}")
print(f"Porcentaje de nulos: {100*nulos_motor/total:.3f}%")

: 

In [ ]:
print(data["Motor"].unique().tolist())

: 

In [ ]:
motor_extraido = (
    data["Motor"]
    .str.extract(r'(\d+[.,]\d+)')[0]
)

perdidos = data[
    data["Motor"].notna() &
    motor_extraido.isna()
]

print(len(perdidos))
display(perdidos["Motor"].value_counts().head(10)) 

: 

<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
En los casos donde la cilindrada no pudo recuperarse, se incorporó la variable indicadora <strong>Motor_Litros_Faltante</strong>, permitiendo al modelo distinguir entre motores con cilindrada conocida y desconocida.

<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
6. <code>Kilómetros</code>: Se transformó a valor numérico para garantizar un tratamiento consistente. Además se crea el feature <code>0km</code> que indica si el vehículo es nuevo o usado. Posteriormente se incorporó <code>Log_Km</code> para reducir la asimetría observada en la distribución original.
</p>

<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
7. <code>Descripción</code>: contiene texto libre que no puede ser utilizado directamente por modelos de regresión tradicionales. Se construyó un score basado en palabras clave positivas y negativas asociadas al estado general del vehículo, documentación, mantenimiento y equipamiento. El resultado se transformó en una escala numérica entre 1 y 10.
</p>

<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
7. <code>Descripción</code>: se aplicó <strong>TF-IDF</strong> (<em>Term Frequency - Inverse Document Frequency</em>) para vectorizar las descripciones, asignando mayor peso a las palabras relevantes y menor a las más frecuentes en todo el dataset. Dado que esto genera un vector de alta dimensionalidad, se aplicó <strong>SVD</strong> para reducir la dimensionalidad a 20 componentes, preservando la mayor parte de la información semántica.
</p>

<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
8. <code>Marca</code>: Se unificaron las muestras para evitar duplicados y/o reconocimiento erróneo de una marca.
A su vez, se combinó <code>Marca</code> y <code>Modelo</code> en una sola feature <code>Marca_Modelo</code>.
</p>


In [ ]:
print(data["Marca"].unique().tolist())

: 

8. `Año`: Considerando que se tiene 1 valor extremo, claro outlier o error de carga, se elimina la muestra ya que no aporta informacion ni se pierde al no tenerla.

In [ ]:
display(pd.DataFrame(data['Año'].describe()))

: 

In [ ]:
print((data["Año"] > 2025).sum()) 
print(data[data["Año"] > 2025]["Año"].unique()) 

: 

<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
9. <code>Color</code>: considerando que se tienen colores definidos en mayuscula y minuscula, femenino y masculino, se unifican todas las muestras a minuscula masculino. A la vez, al haber pocas muestras de colores especificos, tales como 'gris titane' (1 sola muestra) o 'gris artense', el modelo no aprendera de estas, por lo que resulta conveniente unificarlas dentro de los colores genericos (en el caso de los ejemplos, dentro de 'gris). Asi se redujo la cardinalidad sin perder informacion relevante.
</p>

In [ ]:
print(data["Color"].value_counts().to_string())

: 

<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
10. <code>Versión</code>: se agruparon las versiones en tres categroias, Version_Base, Version_Intermedia  y Version_Premium. Se clasificó utilizanod palabras clave frecuentes asociadas al nivel del equipamiento del veehículo.
</p>

<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
11. <code>Con Camara de retroceso</code>: tiene una gran cantidad de valores faltantes. Antes de realizar cualquier imputación estadística, se intentó recuperar esta información buscando referencias a cámaras o sensores de estacionamiento dentro de la descripción de cada vehículo. Cuando no fue posible inferir la presencia de la característica, el registro fue etiquetado como "Desconocido", permitiendo conservar la observación sin introducir supuestos adicionales.
</p>

In [ ]:
data_pre = preprocesamiento_pre_split(data)
print('Tamaño final del dataset -> ', data_pre.shape)

: 

**NUEVO DATASET**

In [ ]:
print("\n Dataset NUEVO, ALEATORIO")
print(data_pre.sample(5))

print("\n -------- RESUMEN - ESTADISTICOS E INFO -------- \n" )
print("\n data_pre INFO")
print(data_pre.info())
print("\n data_pre DESCRIPCION " )
print(data_pre.describe())

: 

In [ ]:
summary = pd.DataFrame({
    "dtype": data_pre.dtypes,
    "nulos": data_pre.isnull().sum(),
    "unicos": data_pre.nunique()
})
summary

: 

In [ ]:
data_pre.to_csv("../data/processed/data_pre__limpio_EDA.csv", index=False)

: 

DETECCION DE OUTLIERS

In [ ]:
df_outliers = data_pre.copy()

: 

In [ ]:
df_outliers = flagear_outliers_por_grupo(df_outliers, grupo_col = "Marca_Modelo", col = "Precio", k = 1.5, min_registros = 10)

: 

In [ ]:
resu = reportar_outliers_por_grupo(df_outliers, grupo_col = "Marca_Modelo", col = "Precio", top_n = 20)
display(resu.round(2))

: 

In [ ]:
ver_outliers(df_outliers, col = "Precio", n = 20)

: 

In [ ]:
vista_outliers_marca= plot_grupo_especifico(df_outliers, grupo="Land Rover_Range Rover Sport", k=1.5)
vista_outliers_marca_= plot_grupo_especifico(df_outliers, grupo="Renault_Duster", k=1.5)
vista_outliers_marca_= plot_grupo_especifico(df_outliers, grupo="Jeep_Wrangler", k=1.5)

: 

*Lo que se logra notar es que hay Precios < 5000 USD los cuales el IQR no logra detectar, tal q se opta por eliminar aquellos precios para evitar outliers por Marca_Modelo.*

In [ ]:
len(df_outliers[df_outliers["Precio"] <= 5000])

: 

kilometros outliers

In [ ]:
display(
    data_pre[data_pre["Kilómetros"] > 500000]
    [["Marca_Modelo", "Precio", "Año", "Kilómetros"]]
    .sort_values("Kilómetros", ascending=False)
)

: 

creo datasets sin outliers

In [ ]:
#elimina llas muestras vuyo precio queda fuera del rango iqr en su marca modelo
data_pre_sin_outliers= eliminar_outliers_grupo(df_outliers)
#elimina km >= 500.000 
data_pre_sin_outliers = data_pre_sin_outliers [data_pre_sin_outliers["Kilómetros"] <= 500000].copy()

print("Original:", data_pre.shape)
print("IQR + KM limpio:", data_pre_sin_outliers.shape)

#elimina precio <5000 USD
data_pre_sin_out_final = eliminar_outliers_por_corte(
    data_pre_sin_outliers,
    precio_min=5000
)

print("IQR + KM + > 5000:", data_pre_sin_out_final.shape)

: 

In [ ]:
data_pre_sin_out_final.to_csv("../data/processed/data_pre_SIN_OUT.csv", index=False)

: 

**VISUALIZACION DEL EDA HASTA AHORA**

In [ ]:
#Plots
eda_visualizacion_suvs(data_pre)

: 

<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
<em>Se observan diferencias de precio según el modelo, el combustible y la condición del vehículo. Además, tanto el kilometraje como la antigüedad presentan una relación negativa con el precio. Las transformaciones logarítmicas permiten visualizar mejor estas relaciones al reducir la asimetría de las distribuciones originales.</em>
</p>

<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
Para terminar con el <em><strong>preprocesamiento de datos</strong></em>, se aplica el split para poder obtener la moda calculada sobre el conjunto de train, y asi poder completar los valores faltantes de las features "color", "transmision", "camara", "Kilómetros", "Motor_Litros" evitando el data leakage.
</p>

In [ ]:
nulos = data_pre.isnull().sum()

if (nulos > 0).any():
    print("Columnas con NaN:")
    print(nulos[nulos > 0])
else:
    print("No quedan NaN en el dataset")

: 

<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
11. <strong>Imputaciones realizadas después del split</strong>: todas las imputaciones basadas en estadísticas (moda o mediana) fueron calculadas exclusivamente sobre el conjunto de entrenamiento y posteriormente aplicadas al conjunto de validación, evitando data leakage.
</p>
<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
- <strong>Color</strong>: los valores faltantes se completaron utilizando la moda del color para cada vehículo (`Marca_Modelo`). Si un modelo no poseía suficientes observaciones, se utilizó la moda global del conjunto de entrenamiento.
</p>
<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
- <strong>Transmisión</strong>: debido a la baja cantidad de valores faltantes, se utilizó un imputador KNN entrenado sobre el conjunto de entrenamiento. La imputación se realizó considerando vehículos con características similares y posteriormente se reconstruyó la categoría original.
</p>
<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
- <strong>Kilómetros</strong>: los valores faltantes se completaron utilizando la mediana del kilometraje correspondiente al año del vehículo. En caso de no existir suficientes observaciones para un determinado año, se utilizó la mediana global del conjunto de entrenamiento.
</p>
<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
- <strong>Motor_Litros</strong>: cuando no fue posible recuperar la cilindrada a partir de la descripción original del motor, el valor faltante se imputó utilizando la mediana calculada sobre el conjunto de entrenamiento. Adicionalmente, se conservó la variable indicadora `Motor_Litros_Faltante` para que el modelo pudiera distinguir los registros originalmente incompletos.
</p>

In [ ]:
train, val = train_val_split(data_pre)

: 

In [ ]:
#Preprocessing post split usando los parametros del entrenamiento
x_train, x_val = preprocesamiento_post_split(train, val)

: 

In [ ]:
#One-Hot Encoding sobre las columnas con baja cardinalidad
columnas_oh = ["Marca_Modelo","Color",'Tipo de combustible', 'Tipo de vendedor', 'Transmisión', 'Con cámara de retroceso']
X_train_final, X_val_final = onehot_encoding(x_train, x_val, columnas_oh)

: 

In [ ]:
print('Tamaño train final -> ', X_train_final.shape)
print('Tamaño val final -> ', X_val_final.shape)

: 

In [ ]:
# print('Final del preprocesamiento sobre Entrenamiento')
# summary = pd.DataFrame({
#     "dtype": X_train_final.dtypes,
#     "nulos": X_train_final.isnull().sum(),
#     "unicos": X_train_final.nunique()
# })
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)

# display(summary)

: 

In [ ]:
# print('Final del preprocesamiento sobre Validación')
# summary_ = pd.DataFrame({
#     "dtype": X_val_final.dtypes,
#     "nulos": X_val_final.isnull().sum(),
#     "unicos": X_val_final.nunique()
# })
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)
# display(summary_)

: 

In [ ]:
#Cargamos el set de entrenamiento y validacion a csv para ver los cambios
X_train_final.to_csv("../data/processed/X_train_EDA.csv", index=False)
X_val_final.to_csv("../data/processed/X_val_EDA.csv", index=False)

: 

### **MODELOS**

In [ ]:
#Sin variables categoricas, considerando OHE
X_train_ohe = X_train_final.drop(columns = ['Precio'])
y_train_ohe = X_train_final['Precio']

X_val_ohe = X_val_final.drop(columns = ['Precio'])
y_val_ohe = X_val_final['Precio']

: 

In [ ]:
#Estandarizacion
#Con 'Descripcion'
X_train_array, X_val_array, _, _ = estandarizar(X_train_ohe.values, X_val_ohe.values)
X_train_ohe_estandarizado = pd.DataFrame(X_train_array, columns = X_train_ohe.columns)
X_val_ohe_estandarizado = pd.DataFrame(X_val_array, columns = X_val_ohe.columns)

#Sin 'Descripcion'
columnas = [col for col in X_train_ohe_estandarizado.columns if col.startswith('Descripcion_')]
X_train_no_desc = X_train_ohe_estandarizado.drop(columns = columnas)
X_val_no_desc = X_val_ohe_estandarizado.drop(columns = [col for col in X_val_ohe_estandarizado.columns if col.startswith('Descripcion_')])

: 

#### **Regresion Lineal** (baseline)

In [ ]:
#Regresion lineal base
modelo_rl, predicciones_rl, _, _, _ = entrenar_regresion_lineal(X_train_ohe_estandarizado, y_train_ohe, X_val_ohe_estandarizado, y_val_ohe)

#Grid search de regularizacion
print(f'Busqueda parametro de regularizacion')

alphas = [0.1, 0.5, 1.0, 10.0]

#Con score de la descripcion
print('Con TF-IDF sobre Descripcion')
resultados_tfidf = definir_regularizacion(X_train_ohe_estandarizado, y_train_ohe, X_val_ohe_estandarizado, y_val_ohe, alphas)
print(f"Mejor R²: {resultados_tfidf['R2'].max():.4f}")

print('SIN TF-IDF sobre Descripcion')
resultados_no_desc = definir_regularizacion(X_train_no_desc, y_train_ohe, X_val_no_desc, y_val_ohe, alphas)
print(f"Mejor R²: {resultados_no_desc['R2'].max():.4f}")

: 

In [ ]:
print('Resultados considerando TF-IDF sobre Descripción')
display(resultados_tfidf)
print('Resultados sin considerar TF-IDF sobre Descripción')
display(resultados_no_desc)

: 

In [ ]:
from sklearn.linear_model import Ridge

mejor_rl = Ridge (alpha=10, solver="svd")
mejor_rl.fit(X_train_ohe_estandarizado, y_train_ohe)

coefs = pd.Series(mejor_rl.coef_, index=X_train_ohe_estandarizado.columns)
desc_coefs = coefs[[c for c in coefs.index if c.startswith("Descripcion_")]]
print(desc_coefs.sort_values(key=abs, ascending=False))

: 

In [ ]:
mejor_tfidf = resultados_tfidf.iloc[0]
mejor_no_desc = resultados_no_desc.iloc[0]

resumen = pd.DataFrame({
    "Configuración": ["Con TF-IDF", "Sin TF-IDF"],
    "Modelo": [mejor_tfidf["Modelo"], mejor_no_desc["Modelo"]],
    "Alpha": [mejor_tfidf["Alpha"], mejor_no_desc["Alpha"]],
    "RMSE (USD)": [round(mejor_tfidf["RMSE"], 2), round(mejor_no_desc["RMSE"], 2)],
    "MAE (USD)": [round(mejor_tfidf["MAE"], 2), round(mejor_no_desc["MAE"], 2)],
    "R²": [round(mejor_tfidf["R2"], 4), round(mejor_no_desc["R2"], 4)]
})
display(resumen)

if mejor_tfidf['R2'] > mejor_no_desc['R2']:
    print(f"Mejor modelo de Regresión Lineal: {mejor_tfidf['Modelo']} con TF-IDF (R²= {mejor_tfidf['R2']:.4f})")
else:
    print(f"Mejor modelode Regresión Lineal: {mejor_no_desc['Modelo']} sin TF-IDF (R²= {mejor_no_desc['R2']:.4f})")

: 

#### **XGBoost**

In [ ]:
#Con variables categoricas
X_train_xgboost = x_train.drop(columns = ['Precio'])
y_train_xgboost = train['Precio']

X_val_xgboost = x_val.drop(columns = ['Precio'])
y_val_xgboost = val['Precio']

print('Con TF-IDF sobre Descripcion')
modelo_xgboost, predicciones_xgboost, _, _, _ = entrenar_xgboost(X_train_xgboost, y_train_xgboost, X_val_xgboost, y_val_xgboost)

print('\nSin TF-IDF sobre Descripcion')
X_train_xgboost_no_desc = X_train_xgboost.drop(columns = columnas)
X_val_xgboost_no_desc = X_val_xgboost.drop(columns = columnas)
modelo_xgboost_no_desc, predicciones_xgboost_no_desc, _, _, _ = entrenar_xgboost(X_train_xgboost_no_desc, y_train_xgboost, X_val_xgboost_no_desc, y_val_xgboost)

: 

In [ ]:
print('Con TF-IDF sobre Descripcion')
xgboost_ohe, xgboost_predicciones_ohe, _, _, _ = entrenar_xgboost_ohe(X_train_ohe, y_train_ohe, X_val_ohe, y_val_ohe)

print('\nSin TF-IDF sobre Descripcion')
X_train_ohe_no_desc = X_train_ohe.drop(columns = columnas)
X_val_ohe_no_desc = X_val_ohe.drop(columns = columnas)

modelo_xgboost_ohe_no_desc, predicciones_xgboost_ohe_no_desc, _, _, _ = entrenar_xgboost_ohe(X_train_ohe_no_desc, y_train_ohe, X_val_ohe_no_desc, y_val_ohe)

: 

### ***SPLIT Y MODELOS CON DATASET SIN OUTLIERS***

In [ ]:
nulos = data_pre_sin_out_final.isnull().sum()

if (nulos > 0).any():
    print("Columnas con NaN:")
    print(nulos[nulos > 0])
else:
    print("No quedan NaN en el dataset")

: 

In [ ]:
train_sin_out, val_sin_out = train_val_split(data_pre_sin_out_final)

: 

In [ ]:
#Preprocessing post split usando los parametros del entrenamiento
x_train_sin_out, x_val_sin_out = preprocesamiento_post_split(train_sin_out, val_sin_out)

: 

In [ ]:
#One-Hot Encoding sobre las columnas con baja cardinalidad
columnas_oh_sin_out = ["Marca_Modelo","Color",'Tipo de combustible', 'Tipo de vendedor', 'Transmisión', 'Con cámara de retroceso']
X_train_final_sin_out, X_val_final_sin_out = onehot_encoding(x_train_sin_out, x_val_sin_out, columnas_oh_sin_out)

: 

In [ ]:
print('Tamaño train final -> ', X_train_final_sin_out.shape)
print('Tamaño val final -> ', X_val_final_sin_out.shape)

: 

### **MODELOS sin outliers**

In [ ]:
#Sin variables categoricas, considerando OHE
X_train_ohe_sin_out = X_train_final_sin_out.drop(columns = ['Precio'])
y_train_ohe_sin_out = X_train_final_sin_out['Precio']

X_val_ohe_sin_out = X_val_final_sin_out.drop(columns = ['Precio'])
y_val_ohe_sin_out = X_val_final_sin_out['Precio']

: 

In [ ]:
#Estandarizacion
#Con 'Descripcion'
X_train_array_sin_out, X_val_array_sin_out, _, _ = estandarizar(X_train_ohe_sin_out.values, X_val_ohe_sin_out.values)
X_train_ohe_estandarizado_sin_out = pd.DataFrame(X_train_array_sin_out, columns = X_train_ohe_sin_out.columns)
X_val_ohe_estandarizado_sin_out = pd.DataFrame(X_val_array_sin_out, columns = X_val_ohe_sin_out.columns)

#Sin 'Descripcion'
columnas_sin_out = [col for col in X_train_ohe_estandarizado_sin_out.columns if col.startswith('Descripcion_')]
X_train_no_desc_sin_out = X_train_ohe_estandarizado_sin_out.drop(columns = columnas_sin_out)
X_val_no_desc_sin_out = X_val_ohe_estandarizado_sin_out.drop(columns = [col for col in X_val_ohe_estandarizado_sin_out.columns if col.startswith('Descripcion_')])

: 

#### **Regresion Lineal** (baseline)

In [ ]:
#Regresion lineal base
modelo_rl, predicciones_rl, _, _, _ = entrenar_regresion_lineal(X_train_ohe_estandarizado_sin_out, y_train_ohe_sin_out, X_val_ohe_estandarizado_sin_out, y_val_ohe_sin_out)

#Grid search de regularizacion
print(f'Busqueda parametro de regularizacion')

alphas = [0.1, 0.5, 1.0, 10.0]

#Con score de la descripcion
#print('Con TF-IDF sobre Descripcion')
resultados_tfidf = definir_regularizacion(X_train_ohe_estandarizado_sin_out, y_train_ohe_sin_out, X_val_ohe_estandarizado_sin_out, y_val_ohe_sin_out, alphas)
#print(f"Mejor R²: {resultados_tfidf['R2'].max():.4f}")

#print('SIN TF-IDF sobre Descripcion')
resultados_no_desc = definir_regularizacion(X_train_no_desc_sin_out, y_train_ohe_sin_out, X_val_no_desc_sin_out, y_val_ohe_sin_out, alphas)
#print(f"Mejor R²: {resultados_no_desc['R2'].max():.4f}")

: 

In [ ]:
mejor_tfidf = resultados_tfidf.iloc[0]
mejor_no_desc = resultados_no_desc.iloc[0]

resumen = pd.DataFrame({
    "Configuración": ["Con TF-IDF", "Sin TF-IDF"],
    "Modelo": [mejor_tfidf["Modelo"], mejor_no_desc["Modelo"]],
    "Alpha": [mejor_tfidf["Alpha"], mejor_no_desc["Alpha"]],
    "RMSE (USD)": [round(mejor_tfidf["RMSE"], 2), round(mejor_no_desc["RMSE"], 2)],
    "MAE (USD)": [round(mejor_tfidf["MAE"], 2), round(mejor_no_desc["MAE"], 2)],
    "R²": [round(mejor_tfidf["R2"], 4), round(mejor_no_desc["R2"], 4)]
})
display(resumen)

if mejor_tfidf['R2'] > mejor_no_desc['R2']:
    print(f"Mejor modelo de Regresión Lineal: {mejor_tfidf['Modelo']} con TF-IDF (R²= {mejor_tfidf['R2']:.4f})")
else:
    print(f"Mejor modelode Regresión Lineal: {mejor_no_desc['Modelo']} sin TF-IDF (R²= {mejor_no_desc['R2']:.4f})")

: 

#### **XGBoost**

In [ ]:
grid = {
    'n_estimator': [100, 300, 500],
    'max_depth': [4,6,8],
    'learning_rate': [0.05, 0.1, 0.5] 
    }

: 

In [ ]:
#Con variables categoricas
X_train_xgboost_sin_out = x_train_sin_out.drop(columns = ['Precio'])
y_train_xgboost_sin_out = train_sin_out['Precio']

X_val_xgboost_sin_out = x_val_sin_out.drop(columns = ['Precio'])
y_val_xgboost_sin_out = val_sin_out['Precio']

modelo_xgboost_sin_out, predicciones_xgboost_sin_out, rmse_xgboost_sin_out, mae_xgboost_sin_out, r2_xgboost_sin_out = entrenar_xgboost(X_train_xgboost_sin_out, y_train_xgboost_sin_out, X_val_xgboost_sin_out, y_val_xgboost_sin_out)

X_train_xgboost_no_desc_sin_out = X_train_xgboost_sin_out.drop(columns = columnas_sin_out)
X_val_xgboost_no_desc_sin_out = X_val_xgboost_sin_out.drop(columns = columnas_sin_out)
modelo_xgboost_no_desc_sin_out, predicciones_xgboost_no_desc_sin_out, rmse_xgboost_no_desc_sin_out, mae_xgboost_no_desc_sin_out, r2_xgboost_no_desc_sin_out = entrenar_xgboost(X_train_xgboost_no_desc_sin_out, y_train_xgboost_sin_out, X_val_xgboost_no_desc_sin_out, y_val_xgboost_sin_out)

: 

In [ ]:
#resumen_xgb = pd.DataFrame({
#    "Configuración": ["Con TF-IDF", "Sin TF-IDF"],
#    "RMSE (USD)": [
#        rmse_xgboost_sin_out,
#        rmse_xgboost_no_desc_sin_out
#    ],
#    "MAE (USD)": [
#        mae_xgboost_sin_out,
#        mae_xgboost_no_desc_sin_out
#    ],
#    "R²": [
#        r2_xgboost_sin_out,
#        r2_xgboost_no_desc_sin_out
#    ]
#})
#display(resumen_xgb)
#
#mejor_xgb = resumen_xgb.loc[resumen_xgb["R²"].idxmax()]
#print(f"Mejor configuración XGBoost categórico: {mejor_xgb['Configuración']} (R²= {mejor_xgb['R²']:.4f})")

: 

In [ ]:
#Con TF-IDF - Grid search categorico
resultados_gs_categorico = grid_search(X_train_xgboost_sin_out, y_train_xgboost_sin_out, n_estimators_list = grid['n_estimator'], max_depth_list = grid['max_depth'], learning_rate_list = grid['learning_rate'])

#Sin TF-IDF - Grid search categorico
resultados_gs_categorico_no_desc = grid_search(X_train_xgboost_no_desc_sin_out, y_train_xgboost_sin_out, n_estimators_list = grid['n_estimator'], max_depth_list = grid['max_depth'], learning_rate_list = grid['learning_rate'])

: 

In [ ]:
xgboost_ohe_sin_out, xgboost_predicciones_ohe_sin_out, rmse_xgboost_ohe_sin_out, mae_xgboost_ohe_sin_out, r2_xgboost_ohe_sin_out = entrenar_xgboost_ohe(X_train_ohe_sin_out, y_train_ohe_sin_out, X_val_ohe_sin_out, y_val_ohe_sin_out)

X_train_ohe_no_desc_sin_out = X_train_ohe_sin_out.drop(columns = columnas_sin_out)
X_val_ohe_no_desc_sin_out = X_val_ohe_sin_out.drop(columns = columnas_sin_out)

modelo_xgboost_ohe_no_desc_sin_out, predicciones_xgboost_ohe_no_desc_sin_out, rmse_xgboost_ohe_no_desc_sin_out, mae_xgboost_ohe_no_desc_sin_out, r2_xgboost_ohe_no_desc_sin_out = entrenar_xgboost_ohe(X_train_ohe_no_desc_sin_out, y_train_ohe_sin_out, X_val_ohe_no_desc_sin_out, y_val_ohe_sin_out)

: 

In [ ]:
#resumen_xgb = pd.DataFrame({
#    "Configuración": ["Con TF-IDF", "Sin TF-IDF"],
#    "RMSE (USD)": [
#        rmse_xgboost_ohe_sin_out,
#        rmse_xgboost_ohe_no_desc_sin_out
#    ],
#    "MAE (USD)": [
#        mae_xgboost_ohe_sin_out,
#        mae_xgboost_ohe_no_desc_sin_out
#    ],
#    "R²": [
#        r2_xgboost_ohe_sin_out,
#        r2_xgboost_ohe_no_desc_sin_out
#    ]
#})
#display(resumen_xgb)
#
#mejor_xgb = resumen_xgb.loc[resumen_xgb["R²"].idxmax()]
#print(f"Mejor configuración XGBoost categórico: {mejor_xgb['Configuración']} (R²= {mejor_xgb['R²']:.4f})")

: 

In [ ]:
#Con TF-IDF - Grid search OHE
resultados_gs_ohe = grid_search(X_train_ohe_sin_out, y_train_xgboost_sin_out, n_estimators_list = grid['n_estimator'], max_depth_list = grid['max_depth'], learning_rate_list = ['learning_rate'], categorico = False)

#Sin TF-IDF - Grid search OHE
resultados_gs_ohe_no_desc = grid_search(X_train_ohe_no_desc_sin_out, y_train_xgboost_sin_out, n_estimators_list = grid['n_estimator'], max_depth_list = grid['max_depth'], learning_rate_list = ['learning_rate'], categorico = False)

: 

In [ ]:
#Guardamos los mejores resultados
mejor_combinacion_categorico = resultados_gs_categorico.iloc[0]
mejor_combinacion_categorico_no_desc = resultados_gs_categorico_no_desc.iloc[0]
mejor_combinacion_ohe = resultados_gs_ohe.iloc[0]
mejor_combinacion_ohe_no_desc = resultados_gs_ohe_no_desc.iloc[0]

: 

In [ ]:
#Entrenamos el mejor modelo para cada caso: Con y Sin TF-IDF y variables categoricas, con y sin TF-IDF con ohe sobre las variables categoricas
#-> Con TF-IDF sobre Descripcion y variables categoricas
modelo_categorico_final, prediccion_categorico_final, rmse_categorico_final, mae_categorico_final, r2_categorico_final = entrenar_xgboost(X_train_xgboost_sin_out, y_train_xgboost_sin_out, X_val_xgboost_sin_out, y_val_xgboost_sin_out, n_estimators = mejor_combinacion_categorico['n_estimator'], max_depth = mejor_combinacion_categorico['max_depth'], learning_rate = mejor_combinacion_categorico['learning rate'])
#-> Sin TF-IDF sobre Descripcion y variables categoricas
modelo_categorico_final_no_desc, prediccion_categorico_final_no_desc, rmse_categorico_final_no_desc, mae_categorico_final_no_desc, r2_categorico_final_no_desc = entrenar_xgboost(X_train_xgboost_no_desc_sin_out, y_train_xgboost_sin_out, X_val_xgboost_no_desc_sin_out, y_val_xgboost_sin_out, n_estimators = mejor_combinacion_categorico_no_desc['n_estimator'], max_depth = mejor_combinacion_categorico_no_desc['max_depth'], learning_rate = mejor_combinacion_categorico_no_desc['learning rate'])
#-> Con TF-IDF sobre Descripcion y OHE
modelo_ohe_final, prediccion_ohe_final, rmse_ohe_final, mae_ohe_final, r2_ohe_final = entrenar_xgboost(X_train_ohe_sin_out, y_train_xgboost_sin_out, X_val_ohe_sin_out, y_val_xgboost_sin_out, n_estimators = mejor_combinacion_ohe['n_estimator'], max_depth = mejor_combinacion_ohe['max_depth'], learning_rate = mejor_combinacion_ohe['learning rate'])
#-> Sin TF-IDF sobre Descripcion y OHE
modelo_ohe_final_no_desc, prediccion_ohe_final_no_desc, rmse_ohe_final_no_desc, mae_ohe_final_no_desc, r2_ohe_final_no_desc = entrenar_xgboost(X_train_ohe_no_desc_sin_out, y_train_xgboost_sin_out, X_val_ohe_no_desc_sin_out, y_val_xgboost_sin_out, n_estimators = mejor_combinacion_categorico_no_desc['n_estimator'], max_depth = mejor_combinacion_categorico_no_desc['max_depth'], learning_rate = mejor_combinacion_categorico_no_desc['learning rate'])

: 

In [ ]:
display(pd.DataFrame({
    'Modelo': ['XGB Categórico Con TF-IDF', 'XGB Categórico Sin TF-IDF', 'XGB OHE Con TF-IDF', 'XGB OHE Sin TF-IDF'],
    'n_estimators': [int(mejor_combinacion_categorico['n_estimators']), int(mejor_combinacion_categorico_no_desc['n_estimators']), int(mejor_combinacion_ohe['n_estimators']), int(mejor_combinacion_ohe_no_desc['n_estimators'])],
    'max_depth': [int(mejor_combinacion_categorico['max_depth']), int(mejor_combinacion_categorico_no_desc['max_depth']), int(mejor_combinacion_ohe['max_depth']), int(mejor_combinacion_ohe_no_desc['max_depth'])],
    'learning_rate': [mejor_combinacion_categorico['learning_rate'], mejor_combinacion_categorico_no_desc['learning_rate'], mejor_combinacion_ohe['learning_rate'], mejor_combinacion_ohe_no_desc['learning_rate']],
    'RMSE (USD)': [rmse_categorico_final, rmse_categorico_final_no_desc, rmse_ohe_final, rmse_ohe_final_no_desc],
    'MAE (USD)': [mae_categorico_final, mae_categorico_final_no_desc, mae_ohe_final, mae_ohe_final_no_desc],
    'R²': [r2_categorico_final, r2_categorico_final_no_desc, r2_ohe_final, r2_ohe_final_no_desc]
}))

: 

### **EXTENSIONES** 

*¿Cuál es el efecto de los años de uso y el kilometraje sobre el precio de un auto?*

In [ ]:
data_visualizacion = pd.concat([X_train_final_sin_out, X_val_final_sin_out], ignore_index = True)
data_visualizacion = data_visualizacion[data_visualizacion["Antiguedad"] <= 50]
plot_precio_segun_rango_ant(data_visualizacion)
plot_precio_segun_antiguedad_km(data_visualizacion)

: 

<p style="text-align: justify; text-justify: inter-word; font-size: 16px;">
Podemos observar, tanto para la <strong>antigüedad</strong> como para el <strong>kilometraje</strong> (plot presentado en la seccion <em>Visualizacion EDA hasta ahora</em>) una clara tendencia. El primer gráfico muestra que a mayor años de uso, el precio del vehículo (en escala logarítmica) decrece de manera consistente. Los autos de 0-2 años concentran los precios más altos, rondando entre los e^10 y e^12 USD, mientras que los de más de 20 años, se ubican en rangos más bajos.
</p>

<p style="text-align: justify; text-justify: inter-word; font-size: 16px;">
El segundo gráfico confirma lo esperado: a mayor log(Km), menor log(Precio). Los vehículos 0km tienen precios más elevados, lo cual es consistente con la realidad; y la dispersión aumenta a medida que sube el kilometraje, lo que genera el desgaste del vehículo, impactando negativamente en su precio.
</p>

<p style="text-align: justify; text-justify: inter-word; font-size: 16px;">
El boxplot por rangos de antiguedad refuerza ambas observaciones: la mediana de precio decae progresivamente, reduciéndose la dispersión en autos más antiguos. Los outliers más extremos se localizan en el rango de los 0-2 años, correspondiendo posiblemente, a vehículos de alta gama 0km.
</p>

<p style="text-align: justify; text-justify: inter-word; font-size: 16px;">
Con esto se deja en evidencia la importancia de ambos features en relación con el precio para la generalización del modelo predictivo a entrenar.
</p>

*¿Hay un ranking de colores de autos en cuanto a precio? ¿Cuál es el precio relativo de los distintos colores?*


In [ ]:
data_unificado = pd.concat([x_train_sin_out, x_val_sin_out]).reset_index(drop = True)

#Mediana para evitar que los otliers afecten
mediana_general = data_unificado['Precio'].median()

precio_por_color = data_unificado.groupby('Color')['Precio'].median().sort_values(ascending = False)
precio_relativo = ((precio_por_color - mediana_general) / mediana_general * 100).round(2)

display(pd.DataFrame({
    'Color': precio_por_color.index,
    'Mediana por color (USD)': precio_por_color.values,
    'Diferencia respecto a la mediana general': precio_relativo.values.round(2).astype(str) + '%'}))

: 

<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
Con esta tabla en consideracion, podemos ver que la existencia de un ranking de colores en cuanto a su precio es real. De esta forma, vehículos con colores menos frecuentes, tales como el <em>rosa</em> y el <em>amarillo</em> son los que presentan la mayor diferencia en cuanto a la mediana general de los precios. En el extremo opuesto, se tienen al <em>verde</em> y <em>dorado</em>, con una diferencia negativa, demostrando que sus precios estan porcentualmente debajo de los generales. Los colores mas comunes como el <em>gris</em>, <em>blanco</em> y <em>negro</em> se mantienen cercanos a la mediana general. Estas diferencias sugieren que el color está correlacionado con el segmento del vehículo: colores como el <em>rosa</em> y el <em>amarillo</em> tienden a aparecer en modelos deportivos o de lujo, mientras que colores atípicos como el <em>verde</em> 
se asocian a vehículos más antiguos o de menor valor de mercado.
</p>

*¿Es más barato comprar un auto a un privado o a una concesionaria/tienda? De ser así, ¿cuál es la diferencia % de precio?*


In [ ]:
#Mediana para evitar que los otliers afecten
mediana_por_vendedor = data_pre_sin_outliers.groupby('Tipo de vendedor')['Precio'].median()
mediana_particulares = mediana_por_vendedor['particular']

valores = ((mediana_por_vendedor - mediana_particulares) / mediana_particulares * 100).round(2)

display(pd.DataFrame({
    'Tipo de vendedor': valores.index,
    'Diferencia respecto a particular': valores.values.round(2).astype(str) + '%'}))

: 

<p style="text-align: justify; text-justify: inter-word; font-size: 17px;">
Se definio la mediana de los precios de los vendedores particulares como la base para la comparacion, razon por la cual se tiene que su valor porcentual es del 0.0%. Los resultados muestran que tanto las concesionarias como las tiendas publican sus vehiculos a precios superiores, siendo estas ultimas las que mayor sobreprecio presentan respecto de los privados. Esto refleja como los costos adicionales que incluyen estos canales tales como garantias y/o tipos de financiacion, afectan sus precios de venta. Más coloquialmente, el precio propuesto por las tiendas es equivalente a la compra de 1 auto y práticamente la mitad de otro.
</p>

In [ ]:
#Definimos que las marcas con mas de 35 muestras sean las validas para calcular el Coeficiente de Variacion
marcas_validas = data['Marca'].value_counts()
marcas_validas = marcas_validas[marcas_validas >= 35].index
data_filtrado = data[data['Marca'].isin(marcas_validas)]

cv = data_filtrado.groupby('Marca')['Precio'].agg(['std', 'mean'])
cv["cv"] = cv["std"] / cv["mean"]
cv.sort_values("cv")

: 

*¿Hay alguna marca que tenga una menor dispersión en sus precios?*

In [ ]:
data_pre_con_marca = data_pre_sin_outliers.copy()
data_pre_con_marca['Marca'] = data['Marca']
data_pre_con_marca = data_pre_con_marca[data_pre_con_marca["Precio"] < 350000]

plot_dispersion_por_marca(data_pre_con_marca, target= "Precio", min_muestras= 35, top_n=12)

: 

<p style="text-align: justify; text-justify: inter-word; font-size: 16px;">
Las marcas con mayor dispersión resultan ser <strong>Land Rover</strong>, <strong>Porsche</strong> y <strong>BMW</strong>, tres marcas de lujo, lo cual explica este resultado: ofrecen modelos que van desde vehículos de entrada hasta vehículos de alta gama, lo que genera una gran variabilidad de precios dentro de sus modelos. El boxplot confirma esto con cajas amplias (particularmente en el caso de Porsche) y outliers con valores extremos.
</p>

<p style="text-align: justify; text-justify: inter-word; font-size: 16px;">
En el extremo opuesto, encontramos a <strong>BAIC</strong>, <strong>Fiat</strong> y <strong>Citroën</strong>, que presentan la menor dispersión entre las marcas más predominantes del dataset. Se presentan como marcas con modelos más accesibles, los cuales se concentran en un rango de precio similar entre sí, y con un catálogo más acotado. Sus boxplots muestran cajas compactas y pocos outliers.
</p>
<p style="text-align: justify; text-justify: inter-word; font-size: 16px;">
Lo observado a partir de la información provista coincide con lo esperable para este tipo de marcas, donde se tienen tanto modelos exclusivos como de segmento medio.
</p>

### Comparación entre modelos

### Final Model